In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy

today = date.today()

In [2]:
def kde_wrapper(data, bw=0.1):
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    
    sns.kdeplot(data, bw_adjust=bw)
    print(describe(data))
    print(mode(data))
    return fig

**Gencode file with expanded regulatory domains (as in GREAT)**

In [4]:
f = open('/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.gtf', 'r')
gencode_file = f.readlines()
f.close()

In [5]:
data = []
for line in gencode_file[5:]:
    line = re.split('\t|;| ', line.strip())
    #gene coordinates
    gene_chrm = line[0]
    gene_start = line[3]
    gene_end = line[4]
    gene_strand = line[6]
    gene_type = line[2]
    
    #gene name
    gene_name_index = line.index('gene_name')
    gene_name = line[gene_name_index + 1].strip("\"")
    
    #gene id (ENSG)
    gene_id_index = line.index('gene_id')
    gene_id = line[gene_id_index + 1].strip("\"")
    #protein id (ENSP)
    if 'protein_id' in line:
        protein_id_index = line.index('protein_id')
        protein_id = line[protein_id_index + 1].strip("\"")
    else:
        protein_id = ''
    #promoter start and end
    if gene_strand == '+':
        promoter_start = gene_start
        promoter_end = str(int(gene_start) + 1)
        basal_start = str(int(gene_start) - 5000)
        basal_end = str(int(gene_start) + 1000)
        domain_start = str(int(gene_start) - 1000000)
        domain_end = str(int(gene_end) + 1000000)
        
    elif gene_strand == '-':
        promoter_start = str(int(gene_end) - 1)
        promoter_end = gene_end
        basal_start = str(int(gene_end) - 1000)
        basal_end = str(int(gene_end) + 5000)
        domain_start = str(int(gene_start) - 1000000)
        domain_end = str(int(gene_end) + 1000000)
        
    data_row = [gene_chrm, gene_start, gene_end, gene_name, gene_type, gene_strand, 
                promoter_start, promoter_end, basal_start, basal_end, domain_start, 
                domain_end, gene_id, protein_id]
    data.append(data_row)

data = np.array(data)

header = ['chrm', 'start', 'end', 'name', 'type', 'strand', 'Promoter start (bp)', 
          'Promoter end (bp)', 'Basal start (bp)', 'Basal end (bp)', 'Domain start (bp)', 
          'Domain end (bp)', 'gene id', 'protein id']

In [6]:
data.shape

(1968859, 14)

In [7]:
#limiting domain to real chromosome positions
chrm = data[:, 0]
tss = data[:, 6].astype(int)
domain_m = data[:, 8:12].astype(int)
names = data[:, 3]

domain_m = np.where(domain_m < 1, 1, domain_m)

In [8]:
domain_m

array([[   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1012227],
       ...,
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023]])

In [9]:
#creating chromosome masked data
chrm_dom_dict = {}
chrm_tss_dict = {}
chrm_name_dict = {}

for chromosome in np.unique(chrm):
    chrm_mask = chrm==chromosome
    chrm_doms = domain_m[chrm_mask, :]
    chrm_tss = tss[chrm_mask]
    chrm_names = names[chrm_mask]
    
    chrm_dom_dict[chromosome] = chrm_doms
    chrm_tss_dict[chromosome] = chrm_tss
    chrm_name_dict[chromosome] = chrm_names

In [43]:
chrm_dom_dict['chr1'].shape

(191060, 4)

In [8]:
chrm_name_dict['chr2']

array(['FAM110C', 'FAM110C', 'FAM110C', ..., 'RPL23AP88', 'RPL23AP88',
       'RPL23AP88'], dtype='<U25')

In [10]:
domain_m

array([[   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1012227],
       ...,
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023]])

In [14]:
gene_dom.shape

(4,)

In [25]:
gene

5655

In [47]:
for gene in range(len(domain_m))[191058:191062]:
    if gene%50000==0:
        print(gene)
    
    gene_chrm = chrm[gene]
    gene_tss = tss[gene]
    gene_name = names[gene]
    gene_dom = domain_m[gene, :]
    
    #filtering out wrong chromosomes
#     chrm_mask = chrm==gene_chrm
#     chrm_doms = domain_m[chrm_mask, :]
#     chrm_tss = tss[chrm_mask]
#     chrm_names = names[chrm_mask]
    
    chrm_doms = chrm_dom_dict[gene_chrm]
    chrm_tss = chrm_tss_dict[gene_chrm]
    chrm_names = chrm_name_dict[gene_chrm]
    print(gene_chrm)
    print(chrm_doms.shape)
    print()
    
    #filtering out other annotations of the same gene
    name_mask = chrm_names!=gene_name
    chrm_name_doms = chrm_doms[name_mask, :]
    chrm_name_tss = chrm_tss[name_mask]
    
    #find all the genes to the left and right
    left_genes = chrm_name_tss<gene_tss
    right_genes = chrm_name_tss>gene_tss
    left_doms = chrm_name_doms[left_genes, :]
    right_doms = chrm_name_doms[right_genes, :]

    #if there are any genes, find the closest basal region
    if len(left_doms) > 0:
        max_left_basal_end = np.max(left_doms[:, 1])
    else:
        max_left_basal_end = gene_dom[2]
    if len(right_doms) > 0:
        min_right_basal_start = np.min(right_doms[:, 0])
    else:
        min_right_basal_start = gene_dom[3]

    #setting the domain for each gene
    gene_dom[2] = np.min([np.max([max_left_basal_end, gene_dom[2]]), gene_dom[0]])
    gene_dom[3] = np.max([np.min([min_right_basal_start, gene_dom[3]]), gene_dom[1]])

chr1
(191060, 4)

chr1
(191060, 4)

chr2
(147486, 4)

chr2
(147486, 4)



In [18]:
domain_m[:gene]

array([[   6869,   12869,       1,   13501],
       [   6869,   12869,       1,   13501],
       [   6869,   12869,       1,   13501],
       ...,
       [2493479, 2499479, 2493479, 2499479],
       [2492853, 2498853, 2492853, 2498853],
       [2525597, 2531597, 2506211, 2531597]])

In [ ]:
#changing domain values
data[:, 8:12] = domain_m.astype(str)

In [15]:
'\t'.join(header)

'chrm\tstart\tend\tname\ttype\tstrand\tPromoter start (bp)\tPromoter end (bp)\tBasal start (bp)\tBasal end (bp)\tDomain start (bp)\tDomain end (bp)\tgene id\tprotein id'

In [ ]:
# outfile = '/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.parsed.promoterbp.extendeddomain.CT20221026.txt'
# np.savetxt(outfile, data, delimiter='\t', fmt='%s', header='\t'.join(header))

In [31]:
f = open('/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.parsed.promoterbp.extendeddomain.CT20221026.txt', 'r')
new_gencode_file = f.readlines()
f.close()

In [15]:
new_gencode_file[1]

'chr1\t11869\t14409\tDDX11L1\tgene\t+\t11869\t11870\t6869\t12869\t1\t13501\tENSG00000223972.5\tNA\n'

In [16]:
line.split('\t')

AttributeError: 'list' object has no attribute 'split'

In [32]:
new_data = []
for line in new_gencode_file[1:]:
    new_data.append(line.strip().split('\t'))
new_data = np.array(new_data)

In [49]:
np.unique(new_data[:, 4])

array(['CDS', 'Selenocysteine', 'UTR', 'exon', 'gene', 'start_codon',
       'stop_codon', 'transcript'], dtype='<U25')

In [33]:
np.sum(new_data[:, 8:12][:gene].astype(int) - domain_m[:gene])

0